<a href="https://colab.research.google.com/github/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/langgraph_101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LangGraph 101 入门指南（面向初学者）

[大语言模型（LLMs）](https://python.langchain.com/docs/concepts/chat_models/) 让我们能够把“智能”嵌入到应用中。[LangGraph](https://langchain-ai.github.io/langgraph/) 是一个用于搭建基于大模型的工作流与智能体（Agent）的框架。本教程以初学者视角，系统介绍 LangGraph 的基础概念、优势、与 [LangChain](https://www.langchain.com/) / [LangSmith](https://docs.smith.langchain.com/) 的配合方式，并给出可运行的最小示例。

![生态系统](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/ecosystem.png?raw=1)

## 聊天模型（Chat Models）

[聊天模型](https://python.langchain.com/docs/concepts/chat_models/) 是多数大模型应用的核心。它们通过“消息”序列作为输入，产出一个“消息”作为输出。LangChain 为聊天模型提供了统一的[标准化接口](https://python.langchain.com/api_reference/langchain/chat_models/langchain.chat_models.base.init_chat_model.html)，便于在[不同厂商](https://python.langchain.com/docs/integrations/chat/)之间切换。

**核心概念（简明版）：**
- **聊天模型**：理解与生成自然语言的 AI 模型。
- **消息（Message）**：带有角色（user/assistant/system）与文本内容的结构化单元。
- **标准化接口**：统一的编程入口，便于替换不同供应商模型。

In [1]:

# 安装必要的依赖包
# 这个命令会安装LangGraph的SQLite检查点器和其他核心依赖
%%capture --no-stderr
# %pip install --quiet -U langgraph-checkpoint-sqlite langchain_core langgraph langchain_openai
%pip install --quiet langchain_openai==0.3.32 langchain_core==0.3.75 langgraph==0.6.7

API易
https://api.apiyi.com/v1
sk-24lYQ4eBJrqcysgU20B1B444674341D493AdA7FcCeD9694f

In [10]:
# 环境变量配置
# 设置OpenAI API密钥，这是使用OpenAI模型所必需的
import os, getpass

def _set_env(var: str):
    """
    安全地设置环境变量
    如果环境变量不存在，会提示用户输入
    """
    # if not os.environ.get(var):
    os.environ[var] = getpass.getpass(f"{var}: ")

# 设置OpenAI API密钥
# 您需要从 https://platform.openai.com/api-keys 获取API密钥
_set_env("OPENAI_API_KEY")
# 设置 OpenAI API代理地址 (例如：https://api.apiyi.com/v1）
_set_env("OPENAI_BASE_URL")

OPENAI_API_KEY: ··········
OPENAI_BASE_URL: ··········


In [11]:
# 初始化聊天模型
# 使用 LangChain 的标准化接口初始化 OpenAI GPT-4.1（可换成你可用的提供商与模型名）
# 提示：temperature=0 输出更稳定、可复现，适合教学与评测
# from langchain.chat_models import init_chat_model
# llm = init_chat_model("openai:gpt-4o", temperature=0)
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")

## 运行模型

`init_chat_model` 返回的对象遵循 [Runnable 标准](https://python.langchain.com/docs/concepts/runnables/)，常用方法：

- `invoke()`：单次请求-响应（同步）。
- `stream()`：按 token/片段流式返回，适合实时展示。

**选择建议：**
- 使用 `invoke()` 进行一次性问答或批处理。
- 使用 `stream()` 构建有“逐字生成”体验的聊天或文案编辑器。

In [13]:
# 调用模型进行问答
# 使用 invoke 方法向模型提问，获取关于"智能体"的解释
result = llm.invoke("什么是智能体？")

In [14]:
# 查看返回结果的类型
# 了解模型返回的数据结构
type(result)

langchain_core.messages.ai.AIMessage

In [15]:
# 使用 Rich 库美化显示结果
# 将模型返回的文本内容以 Markdown 格式渲染显示
from rich.markdown import Markdown
Markdown(result.content)

智能体（Agent）在计算机科学和人工智能领域中，通常指的是一个具有自主行为能力的系统或实体。智能体可以感知其环境，通过
决策和行动影响环境，并能够自动执行特定任务。根据不同的应用，智能体可以具备不同程度的智能，表现出简单的反应行为或复 
杂的推理和学习能力。                                                                                               

智能体的基本特征包括：                                                                                             

 1 自主性：能够在不直接依赖外部指令的情况下进行操作和决策。                                                        
 2 感知能力：能够通过传感器获取环境信息。                                                                          
 3 决策能力：能够基于感知到的信息和内置策略进行判断和选择。                                                        
 4 行为能力：能够采取动作以影响环境或完成任务。                                                                    
 5 适应性：能够随着环境变化或经验的积累进行调整和学习。                                                            

智能体的应用领域广泛，包括机器人、自主车辆、虚拟助手、游戏对手、智能家居系统等。在这些应用中，智能体可能会利用各种 
人工智能技术，如机器学习、自然语言处理和计算机视觉，以提高其决策和互动能力。

## 工具（Tools）

[工具](https://python.langchain.com/docs/concepts/tools/) 是模型可调用的外部能力。在 LangChain 中，可用 `@tool` 将普通 Python 函数“声明”为工具。装饰器会自动基于函数签名与 docstring 推断工具名、描述、参数。

也可采用基于 MCP（Model Context Protocol）的工具方式（见 `https://github.com/langchain-ai/langchain-mcp-adapters`）。

**要点：**
- **工具的作用**：把“会说话”的模型变成“能办事”的系统（如发邮件、查数据库、调外部 API）。
- **装饰器声明**：最小改动即可把函数暴露给模型。
- **结构化元数据**：自动推断有助于提升调用可靠性。

In [16]:
# 创建工具示例
# 使用 @tool 装饰器将 Python 函数转换为 AI 可调用的工具
from langchain.tools import tool

@tool
def write_email(to: str, subject: str, content: str) -> str:
    """
    编写并发送邮件。

    参数:
        to: 收件人邮箱地址
        subject: 邮件主题
        content: 邮件内容

    返回:
        发送确认信息
    """
    # 占位符响应 - 在实际应用中会真正发送邮件
    return f"Email sent to {to} with subject '{subject}' and content: {content}"

In [17]:
# 查看工具对象的类型
# 了解 @tool 装饰器如何转换函数
type(write_email)

langchain_core.tools.structured.StructuredTool

In [18]:
# 查看工具的参数信息
# 了解工具需要哪些输入参数
write_email.args

{'to': {'title': 'To', 'type': 'string'},
 'subject': {'title': 'Subject', 'type': 'string'},
 'content': {'title': 'Content', 'type': 'string'}}

In [19]:
# 查看工具的描述信息
# 了解工具的功能说明
Markdown(write_email.description)

编写并发送邮件。                                                                                                   

参数: to: 收件人邮箱地址 subject: 邮件主题 content: 邮件内容                                                       

返回: 发送确认信息

## 工具调用（Tool Calling）

当工具与模型“绑定”后，模型可在回答中返回“调用某个工具及其参数”的结构化指令，驱动外部动作（参考：[概念](https://python.langchain.com/docs/concepts/tool_calling/)）。使用 `bind_tools` 即可为 LLM 增加调用能力。

![工具调用详情](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/tool_call_detail.png?raw=1)

多数提供商支持 `tool_choice`（参考：[用法](https://python.langchain.com/docs/how_to/tool_choice/)）：
- 指定具体工具名：强制调用该工具。
- 设为 `any`：鼓励至少调用一个工具。

并发控制：将 `parallel_tool_calls=False` 可限制一次仅调用一个工具（参考：[并行调用](https://python.langchain.com/docs/how_to/tool_calling_parallel/)）。

**关键点回顾：**
- **绑定工具**：让模型“看见并可选”这些工具。
- **结构化输出**：包含 `tool` 与 `args`，便于可靠解析与执行。
- **选择策略**：结合 `tool_choice` 与系统提示，引导模型何时用工具。
- **并发与顺序**：根据幂等性与资源限制设置是否并行。

In [20]:
# 将工具绑定到聊天模型
# 使用 bind_tools 方法为模型添加工具能力
model_with_tools = llm.bind_tools([write_email], tool_choice="any", parallel_tool_calls=False)

# 现在模型可以调用工具了
# 向模型发送请求，让它使用工具来完成任务
output = model_with_tools.invoke("起草一份关于明天会议的回复给我的老板（boss@company.ai）")

In [21]:
# 查看模型输出的类型
# 了解工具调用后的返回结果结构
type(output)

langchain_core.messages.ai.AIMessage

In [22]:
# 查看完整的模型输出
# 包含工具调用的详细信息
output

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_zV87DHm1m5dmwSktqelhRGx2', 'function': {'arguments': '{"to":"boss@company.ai","subject":"关于明天会议的回复","content":"尊敬的老板，\\n\\n感谢您关于明天会议的通知。我已收到会议安排并确认参加。请问是否有需要提前准备的材料或需要讨论的具体议题？如有任何需要我准备的内容，请随时告知。\\n\\n期待在会议中与您探讨详细内容。\\n\\n谢谢！\\n\\n致敬\\n\\n[您的名字]"}', 'name': 'write_email', 'parameters': None}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 114, 'prompt_tokens': 103, 'total_tokens': 217, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'input_tokens': 0, 'output_tokens': 0, 'input_tokens_details': None}, 'model_name': 'gpt-4o', 'system_fingerprint': 'fp_ee1d74bde0', 'id': 'chatcmpl-CGPVGgv6sLhCtbDNvbyx1SUgUjpX7', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--9d32077e-8820-4643-ba98-fbe

In [23]:
# 提取工具调用参数并执行工具
# 从模型输出中获取工具调用的参数
args = output.tool_calls[0]['args']
args

{'to': 'boss@company.ai',
 'subject': '关于明天会议的回复',
 'content': '尊敬的老板，\n\n感谢您关于明天会议的通知。我已收到会议安排并确认参加。请问是否有需要提前准备的材料或需要讨论的具体议题？如有任何需要我准备的内容，请随时告知。\n\n期待在会议中与您探讨详细内容。\n\n谢谢！\n\n致敬\n\n[您的名字]'}

In [24]:
# 调用工具执行任务
# 使用提取的参数调用 write_email 工具
result = write_email.invoke(args)
Markdown(result)

Email sent to boss@company.ai with subject '关于明天会议的回复' and content: 尊敬的老板，                          

感谢您关于明天会议的通知。我已收到会议安排并确认参加。请问是否有需要提前准备的材料或需要讨论的具体议题？如有任何需 
要我准备的内容，请随时告知。                                                                                       

期待在会议中与您探讨详细内容。                                                                                     

谢谢！                                                                                                             

致敬                                                                                                               

[您的名字]

![基础提示](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/tool_call.png?raw=1)

## 工作流（Workflows）

构建大模型应用常见有两类模式。

[可以把 LLM 的调用嵌入到预定义的「工作流」中](https://langchain-ai.github.io/langgraph/tutorials/workflows/)，让系统按设定流程有序推进，也可在流程中加入“路由判断”。

例如，我们可以添加一个路由步骤来决定是否需要编写邮件。

![工作流示例](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/workflow_example.png?raw=1)

## 智能体（Agents）

进一步地，我们可以赋予系统更多自主性，让 LLM 动态选择与使用工具。

[智能体](https://langchain-ai.github.io/langgraph/tutorials/workflows/#agent)通常表现为“循环中的工具调用”：每次工具输出会影响下一步决策。

![智能体示例](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/agent_example.png?raw=1)

- 当任务开放、步骤难以事先穷举时，优选智能体。
- 当控制流可提前定义时，更适合工作流。

![工作流与智能体对比](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/workflow_v_agent.png?raw=1)

**工作流 vs 智能体：**
- **工作流**：预定义步骤序列，清晰、可控、易测试。
- **智能体**：动态决策与工具选择，适合复杂或探索式任务。
- **选择建议**：优先选简单可控方案；仅在需要灵活探索时引入智能体。

## 什么是 LangGraph？

[LangGraph](https://langchain-ai.github.io/langgraph/concepts/high_level/) 为任意工作流或智能体提供低层基础设施。

它不对提示词或整体架构做“强抽象”，但带来以下价值：

- **可控性（Control）**：易于定义与组合工作流/智能体。
- **持久化（Persistence）**：图状态可持久化，从而支持记忆与人类介入（HITL）。
- **测试/调试/部署**：提供便捷的测试、调试与部署通道。

### 控制（Control）

LangGraph 将应用抽象为一张图，包含：

1. **状态（State）**：在应用过程中需要持续跟踪的信息。
2. **节点（Nodes）**：如何在流程中更新这些信息。
3. **边（Edges）**：如何连接各个节点以形成控制流。

使用 [`StateGraph` 类](https://langchain-ai.github.io/langgraph/concepts/low_level/#graphs) 可用一个 [`State` 对象](https://langchain-ai.github.io/langgraph/concepts/low_level/#state) 初始化图。

`State` 用于定义贯穿应用生命周期的“状态模式”。

在 Python 中，满足 `getattr()` 的多种对象均可作为 State，例如：

- `TypedDict`：速度最快，但不支持默认值。
- `dataclass`：几乎同样快速，支持点语法 `state.foo`，且可设默认值。
- `pydantic`：较慢（尤其自定义校验器时），但提供类型校验能力。

In [25]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class StateSchema(TypedDict):
    request: str
    email: str

workflow = StateGraph(StateSchema)

每个节点本质上就是一段可执行代码（如 Python 函数）。这让我们能完全掌控节点内部的业务逻辑。

节点接收当前状态，返回一个字典来更新状态。

默认情况下，[相同键的状态会被覆盖](https://langchain-ai.github.io/langgraph/how-tos/state-reducers/)。

如需自定义“合并策略”，可以[定义自定义的更新逻辑（Reducer）](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers)。

![nodes_edges](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/nodes_edges.png?raw=1)

In [26]:
def write_email_node(state: StateSchema) -> StateSchema:
    # 处理输入请求的命令式代码示例
    output = model_with_tools.invoke(state["request"])
    args = output.tool_calls[0]['args']
    email = write_email.invoke(args)
    return {"email": email}

边（Edges）用于连接节点。

我们通过向状态图添加节点与边来指定控制流。

In [27]:
workflow = StateGraph(StateSchema)
workflow.add_node("write_email_node", write_email_node)
workflow.add_edge(START, "write_email_node")
workflow.add_edge("write_email_node", END)

app = workflow.compile()

In [28]:
app.invoke({"request": "起草一份关于明天会议的回复给我的老板（boss@company.ai）"})

{'request': '起草一份关于明天会议的回复给我的老板（boss@company.ai）',
 'email': "Email sent to boss@company.ai with subject 'Re: 明天的会议' and content: 尊敬的老板，\n\n感谢您的通知。我已收到关于明天会议的安排，并会按时出席。\n\n请问会议前是否有需要提前准备的材料或需要我进一步关注的事项？\n\n期待在会面中一起探讨更多。\n\n祝好！\n\n[你的名字]"}

节点间的路由可以通过一个简单函数来[按条件进行](https://langchain-ai.github.io/langgraph/concepts/low_level/#conditional-edges)。

该函数的返回值将作为后继节点（或节点列表）的名称，用于决定状态应被传递到哪里。

可选地，你可以提供一个字典，把 `should_continue` 的返回值映射到具体的下游节点名称。

In [29]:
from typing import Literal
from langgraph.graph import MessagesState
from email_assistant.utils import show_graph

def call_llm(state: MessagesState) -> MessagesState:
    """运行 LLM，一次生成一条消息"""

    output = model_with_tools.invoke(state["messages"])
    return {"messages": [output]}

def run_tool(state: MessagesState):
    """按模型给出的工具调用，逐一执行工具"""

    result = []
    for tool_call in state["messages"][-1].tool_calls:
        observation = write_email.invoke(tool_call["args"])
        result.append({"role": "tool", "content": observation, "tool_call_id": tool_call["id"]})
    return {"messages": result}

def should_continue(state: MessagesState) -> Literal["run_tool", "__end__"]:
    """是否继续：若存在工具调用则转入工具执行，否则结束（回复用户）"""

    # 取最后一条消息
    messages = state["messages"]
    last_message = messages[-1]

    # 若最后一条包含工具调用，则进入工具执行
    if last_message.tool_calls:
        return "run_tool"
    # 否则结束（返回给用户）
    return END

workflow = StateGraph(MessagesState)
workflow.add_node("call_llm", call_llm)
workflow.add_node("run_tool", run_tool)
workflow.add_edge(START, "call_llm")
workflow.add_conditional_edges("call_llm", should_continue, {"run_tool": "run_tool", END: END})
workflow.add_edge("run_tool", END)

# Run the workflow
app = workflow.compile()

ModuleNotFoundError: No module named 'email_assistant'

In [ ]:
show_graph(app)

In [ ]:
result = app.invoke({"messages": [{"role": "user", "content": "Draft a response to my boss (boss@company.ai) confirming that I want to attend Interrupt!"}]})
for m in result["messages"]:
    m.pretty_print()

借助这些低层组件，你可以构建多种多样的工作流与智能体。参见[这份教程](https://langchain-ai.github.io/langgraph/tutorials/workflows/)。

由于智能体非常常见，[LangGraph](https://langchain-ai.github.io/langgraph/tutorials/workflows/#pre-built) 提供了[预构建的智能体抽象](https://langchain-ai.github.io/langgraph/agents/overview/?ref=blog.langchain.dev#what-is-an-agent)。

使用 LangGraph 的[预构建方法](https://langchain-ai.github.io/langgraph/tutorials/workflows/#pre-built)时，你只需传入 LLM、工具与提示词即可。

In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    model=llm,
    tools=[write_email],
    prompt="Respond to the user's request using the tools provided."
)

# 运行智能体
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Draft a response to my boss (boss@company.ai) confirming that I want to attend Interrupt!"}]}
)

for m in result["messages"]:
    m.pretty_print()

### 持久化（Persistence）

#### 线程（Threads）

在长时任务中，让智能体“暂停并稍后继续”通常很有用。

LangGraph 内置持久化层（通过检查点器 Checkpointer 实现）以支持该能力。

当你在编译图时提供检查点器，每一步都会保存图状态的[检查点](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpoints)。

检查点归属于某个“线程（thread）”，在执行结束后也可读取该线程的状态。

![checkpointer](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/checkpoints.png?raw=1)

下面我们以一个[内存检查点器](https://langchain-ai.github.io/langgraph/concepts/persistence/#checkpointer-libraries)为例。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

agent = create_react_agent(
    model=llm,
    tools=[write_email],
    prompt="Respond to the user's request using the tools provided.",
    checkpointer=InMemorySaver()
)

# 线程（thread）标识用于持续对话状态
config = {"configurable": {"thread_id": "1"}}
result = agent.invoke({"messages": [{"role": "user", "content": "What are some good practices for writing emails?"}]}, config)


In [ ]:
# 获取最新的状态快照
config = {"configurable": {"thread_id": "1"}}
state = agent.get_state(config)
for message in state.values['messages']:
    message.pretty_print()

In [ ]:
# 继续对话
result = agent.invoke({"messages": [{"role": "user", "content": "Good, let's use lesson 3 to craft a response to my boss confirming that I want to attend Interrupt"}]}, config)
for m in result['messages']:
    m.pretty_print()

In [ ]:
# 继续对话
result = agent.invoke({"messages": [{"role": "user", "content": "I like this, let's write the email to boss@company.ai"}]}, config)
for m in result['messages']:
    m.pretty_print()

#### 中断（Interrupts）

在 LangGraph 中，也可以通过[中断](https://langchain-ai.github.io/langgraph/how-tos/human_in_the_loop/wait-user-input/)在特定位置暂停图的执行。

常见用法是：向用户收集补充信息，随后继续执行。

In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

from langgraph.types import Command, interrupt
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    input: str
    user_feedback: str

def step_1(state):
    print("---Step 1---")
    pass

def human_feedback(state):
    print("---human_feedback---")
    feedback = interrupt("Please provide feedback:")
    return {"user_feedback": feedback}

def step_3(state):
    print("---Step 3---")
    pass

builder = StateGraph(State)
builder.add_node("step_1", step_1)
builder.add_node("human_feedback", human_feedback)
builder.add_node("step_3", step_3)
builder.add_edge(START, "step_1")
builder.add_edge("step_1", "human_feedback")
builder.add_edge("human_feedback", "step_3")
builder.add_edge("step_3", END)

# 设置持久化存储（内存检查点器）
memory = InMemorySaver()

# 编译图并接入检查点器
graph = builder.compile(checkpointer=memory)

In [ ]:
show_graph(graph)

In [ ]:
# 输入示例
initial_input = {"input": "hello world"}

# 线程（thread）配置
thread = {"configurable": {"thread_id": "1"}}

# 运行图直至第一次中断
for event in graph.stream(initial_input, thread, stream_mode="updates"):
    print(event)
    print("\n")

要从中断处恢复，可以使用[`Command` 对象](https://langchain-ai.github.io/langgraph/how-tos/command/)。

我们向 `resume` 传入一个值，用于作为 `interrupt` 调用的返回值，从而让图从暂停的状态继续执行。

In [ ]:
# 继续执行图（从中断处恢复）
for event in graph.stream(
    Command(resume="go to step 3!"),
    thread,
    stream_mode="updates",
):
    print(event)
    print("\n")

### 追踪（Tracing）

使用 LangChain 或 LangGraph 时，只需设置如下环境变量，即可让 LangSmith 日志[开箱即用](https://docs.smith.langchain.com/observability/how_to_guides/trace_with_langgraph)：

```
export LANGSMITH_TRACING=true
export LANGSMITH_API_KEY="<your-langsmith-api-key>"
```

这是上文智能体执行的一条 LangSmith 追踪示例：

https://smith.langchain.com/public/6f77014f-d054-44ed-aa2c-8b06ceab689f/r

可以看到，智能体能从先前的图状态继续对话，这是因为启用了检查点器（持久化）。

### 部署（Deployment）

可以使用 [LangGraph Platform](https://langchain-ai.github.io/langgraph/concepts/langgraph_platform/) 部署你的图。

它会创建一个带交互式 IDE（LangGraph [Studio](https://langchain-ai.github.io/langgraph/concepts/langgraph_studio/)）与[API](https://langchain-ai.github.io/langgraph/cloud/reference/api/api_ref.html) 的服务，便于以 HTTP 与可视化方式进行交互与调试。

项目需具备如下[结构](https://langchain-ai.github.io/langgraph/concepts/application_structure/)：

```
my-app/
├── src/email_assistant # 业务代码
│   └── langgraph101.py # 构建你的图的代码
├── .env # 环境变量
├── langgraph.json  # LangGraph 配置
└── pyproject.toml # 依赖定义
```

`langgraph.json` 用于声明依赖、图的入口、环境变量等，便于启动 LangGraph 服务。

例如，要部署 `langgraph_101.py`，此仓库里已在 `langgraph.json` 中声明：

```
 "langgraph101": "./src/email_assistant/langgraph_101.py:app",
```

部署选项（参考[文档](https://langchain-ai.github.io/langgraph/tutorials/deployment/)）：
- 本地：在仓库根目录运行 `langgraph dev`。检查点保存到本地文件系统。
- 自托管：多种方式可选。
- 托管：检查点保存到 Postgres（通过 Postgres 检查点器）。

测试提示示例：
```
Draft a response to my boss (boss@company.ai) confirming that I want to attend Interrupt!
```

在 Studio 中可以同时看到图的可视化与当前图状态。

![langgraph_studio](https://github.com/FlyAIBox/langgraph-agents-from-scratch/blob/fly101/notebooks/img/langgraph_studio.png?raw=1)

本地部署的 API 文档可在此处查看：

http://127.0.0.1:2024/docs